# Market data

> Statements whose relations are known, for training and testing a GLiNER2 model to find dependencies between prediction markets: Paleka et al.'s forecasting questions with their negations, paraphrases, consequences and combinations; exclusive groups and numeric ladders from Kalshi, Polymarket and Manifold; and a generator of market questions whose every relation follows from the outcomes it can have.

In [ ]:
#| default_exp markets

In [ ]:
#| export
import hashlib, json, random, re, time, urllib.error, urllib.parse, urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from itertools import product

from sysone.multitask import statement_list, MARKET_WORLD, MARKET_KINDS, _part, _money, _unique

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
from sysone.multitask import gold_example, spans_ok

Two prediction markets depend on each other when one's outcome constrains the other's: if "Bitcoin above $100,000 on June 30" resolves yes, "above $90,000" must too (the first *implies* the second); "Lula wins" and "Zema wins" cannot both resolve yes (each *rules out* the other). A model that marks these relations between the questions of a list builds the dependency graph of a market in a few passes ([GLiNER multitask](33_gliner_multitask.ipynb#statements-and-how-they-relate)).

Training it needs lists whose relations are all known: every pair in a list is either marked or taught as unrelated, so a pair whose relation nobody knows must never share a list. Here every source becomes *groups*, statements whose relations are all known, each with the *keys* of what it is about; `group_lists` joins groups into lists only when nothing could relate them.

In [ ]:
#| export
_STOP = set("""Will Would Is Are Does Do Did Has Have Who What Which When Where How Why The A An In On At By Of For To And Or Not Before After Between
January February March April May June July August September October November December Jan Feb Mar Apr Jun Jul Aug Sep Sept Oct Nov Dec
Monday Tuesday Wednesday Thursday Friday Saturday Sunday Q1 Q2 Q3 Q4 Yes No Other Any""".split())

def norm_key(text:str) -> str:
    "A statement's identity for keeping it apart: lowercase letters and digits"
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def names(text:str) -> set:
    "The capitalised words of a statement (names of people, places, teams, companies), without question words, months and the like"
    return {w for w in re.findall(r"\b[A-Z][\w'’.-]*", text) if w.strip(".") not in _STOP}

def group(statements:list, pairs:list, keys, source:str, id:str, **meta) -> dict:
    "A group: statements whose every relation is known (`pairs`, (i, j, relation), the rest unrelated), and the keys of what it is about"
    return {"id": id, "source": source, "statements": list(statements), "pairs": list(pairs), "keys": set(keys), "names": set().union(*map(names, statements))} | meta

## Paleka et al.'s forecasting questions

[Paleka et al. (2024)](https://arxiv.org/abs/2412.18544) built [tuples of forecasting questions](https://huggingface.co/datasets/dpaleka/ccflmf) whose logical relations are fixed by construction: a base question P from Metaculus, Manifold, news or the year 2028, and questions an LLM wrote from it and a second LLM call checked. The relations each kind of tuple fixes:

| tuple | statements | relations |
|---|---|---|
| Neg | P, not P | each rules out the other |
| Paraphrase | P, P reworded | each implies the other |
| Consequence | P, a consequence of P | P implies it |
| And | P, Q, P and Q | P and Q implies P, and implies Q |
| Or | P, Q, P or Q | P implies P or Q, and so does Q |
| AndOr | P, Q, P and Q, P or Q | P and Q implies P, Q and P or Q; P and Q each imply P or Q |

How P and Q relate in the last three is not given (two candidates for one nomination rule each other out; two unrelated questions do not), so P and Q never share a group.

In [ ]:
#| export
def _title(q) -> str: return " ".join(((q or {}).get("title") or "").split())       # a few titles hold tabs and line breaks

def paleka_groups(rows, check:str, source:str="paleka") -> list:
    "Groups from one kind of Paleka et al.'s tuples (`check`: Neg, Paraphrase, Consequence, And, Or, AndOr), the questions' titles as statements; P and Q of a tuple never share a group"
    out = []
    for i, r in enumerate(rows):
        P = _title(r.get("P")); Q = _title(r.get("Q"))
        if check == "Neg": gs = [([P, _title(r.get("not_P"))], [(0, 1, "rules out")])]
        elif check == "Paraphrase": gs = [([P, _title(r.get("para_P"))], [(0, 1, "implies"), (1, 0, "implies")])]
        elif check == "Consequence": gs = [([P, _title(r.get("cons_P"))], [(0, 1, "implies")])]
        elif check == "And": A = _title(r.get("P_and_Q")); gs = [([A, P], [(0, 1, "implies")]), ([A, Q], [(0, 1, "implies")])]
        elif check == "Or": O = _title(r.get("P_or_Q")); gs = [([P, O], [(0, 1, "implies")]), ([Q, O], [(0, 1, "implies")])]
        elif check == "AndOr":
            A, O = _title(r.get("P_and_Q")), _title(r.get("P_or_Q"))
            gs = [([A, X, O], [(0, 1, "implies"), (1, 2, "implies"), (0, 2, "implies")]) for X in (P, Q)]
        else: raise ValueError(f"no Paleka check {check!r}")
        keys = {norm_key(x) for x in (P, Q) if x}
        for k, (st, pairs) in enumerate(gs):
            if all(st) and len({norm_key(s) for s in st}) == len(st):
                out.append(group(st, pairs, keys, source, f"{source}/{check}/{i}/{k}", check=check))
    return out

In [ ]:
cons = [{"P": {"title": "Will the 100m winning time break 9.63s?"}, "cons_P": {"title": "Will the 100m winning time break 9.70s?"}}]
andt = [{"P": {"title": "Will Haley be the Republican nominee?"}, "Q": {"title": "Will DeSantis be the Republican nominee?"},
         "P_and_Q": {"title": "Will both Haley and DeSantis be the Republican nominees?"}}]
test_eq(paleka_groups(cons, "Consequence")[0]["pairs"], [(0, 1, "implies")])
a = paleka_groups(andt, "And")
test_eq([g["statements"][1] for g in a], ["Will Haley be the Republican nominee?", "Will DeSantis be the Republican nominee?"])  # P and Q apart
assert all(g["keys"] == a[0]["keys"] for g in a) and a[0]["names"] >= {"Haley", "Republican"}

## Exclusive groups and ladders from the exchanges

The exchanges fix some relations by their own rules of settlement:

- **Exclusive groups.** A Kalshi event marked `mutually_exclusive`, a Polymarket event with `negRisk` ("exactly one market in the group resolves Yes"), a Manifold multiple-choice market whose answers sum to one: at most one of its markets resolves yes, so each rules out every other.
- **Ladders.** Kalshi markets that settle on a number carry its bounds (`strike_type` greater, greater or equal, less, less or equal, between; `floor_strike`, `cap_strike`): two markets of one event resolve on the same number, so one implies another when its range lies inside the other's, and they rule each other out when the ranges do not meet. A Polymarket event whose questions differ in one number ("reach $100,000", "reach $120,000") or one date ("by June", "by December") is a ladder too, its direction read from words such as *above*, *reach*, *below*, *by*.

What they do not fix is how markets of different events relate, which is what a dependency graph is mostly about; those lists only join events that share no name.

In [ ]:
#| export
def _interval(m:dict):
    "The values a Kalshi market settles yes on, from its strike: (low, low included, high, high included); None when it has no numeric strike"
    t, lo, hi = m.get("strike_type"), m.get("floor_strike"), m.get("cap_strike")
    inf = float("inf")
    if t == "greater" and lo is not None: return (float(lo), False, inf, False)
    if t == "greater_or_equal" and lo is not None: return (float(lo), True, inf, False)
    if t == "less" and hi is not None: return (-inf, False, float(hi), False)
    if t == "less_or_equal" and hi is not None: return (-inf, False, float(hi), True)
    if t == "between" and lo is not None and hi is not None: return (float(lo), True, float(hi), True)
    return None

def _inside(a, b) -> bool:
    "Whether interval a lies within interval b"
    lo_ok = a[0] > b[0] or (a[0] == b[0] and (b[1] or not a[1]))
    hi_ok = a[2] < b[2] or (a[2] == b[2] and (b[3] or not a[3]))
    return lo_ok and hi_ok

def _apart(a, b) -> bool:
    "Whether two intervals share no value"
    first, second = (a, b) if (a[0], not a[1]) <= (b[0], not b[1]) else (b, a)
    return first[2] < second[0] or (first[2] == second[0] and not (first[3] and second[1]))

def _sample(xs:list, most:int, rng) -> list:
    return list(xs) if len(xs) <= most else (rng or random).sample(list(xs), most)

def kalshi_group(e:dict, most:int=8, rng=None, source:str="kalshi") -> dict|None:
    "A group from a Kalshi event (with its markets): a ladder when every market settles on a number, else an exclusive group when the event is mutually exclusive; None otherwise"
    ms = [m for m in e.get("markets") or [] if (m.get("title") or "").strip()]
    titles = [m["title"].strip() for m in ms]
    if len(set(titles)) < len(titles):                    # one title for every market: the market's own label tells them apart
        titles = [f"{t.rstrip('?')}: {m.get('yes_sub_title') or m.get('subtitle') or ''}".strip(": ") for t, m in zip(titles, ms)]
    keep = [i for i in _sample(range(len(ms)), most, rng)]
    ivs = [_interval(ms[i]) for i in keep]
    st = [titles[i] for i in keep]
    if len(st) < 2 or len(set(st)) < len(st): return None
    if all(ivs):
        pairs = [(a, b, "implies") for a in range(len(st)) for b in range(len(st)) if a != b and _inside(ivs[a], ivs[b])]
        pairs += [(a, b, "rules out") for a in range(len(st)) for b in range(len(st)) if a < b and _apart(ivs[a], ivs[b])]
        kind = "ladder"
    elif e.get("mutually_exclusive"): pairs, kind = [(a, b, "rules out") for a in range(len(st)) for b in range(a + 1, len(st))], "exclusive"
    else: return None
    return group(st, pairs, {e.get("event_ticker", "")}, source, f"{source}/{e.get('event_ticker', '')}", kind=kind, series=e.get("series_ticker"),
                 event=e.get("event_ticker"), category=e.get("category"))

In [ ]:
ladder = {"event_ticker": "KXCHAMPS-30", "series_ticker": "KXCHAMPS", "mutually_exclusive": False, "markets": [
    {"title": "At least 2 championships before 2030?", "strike_type": "greater_or_equal", "floor_strike": 2},
    {"title": "At least 3 championships before 2030?", "strike_type": "greater_or_equal", "floor_strike": 3},
    {"title": "Fewer than 2 championships before 2030?", "strike_type": "less", "cap_strike": 2}]}
g = kalshi_group(ladder)
test_eq(sorted(g["pairs"]), [(0, 2, "rules out"), (1, 0, "implies"), (1, 2, "rules out")])
temps = {"event_ticker": "KXHIGHNY-25JUN03", "mutually_exclusive": True, "markets": [
    {"title": "High temperature in NYC on Jun 3?", "yes_sub_title": "80° to 81°", "strike_type": "between", "floor_strike": 80, "cap_strike": 81},
    {"title": "High temperature in NYC on Jun 3?", "yes_sub_title": "82° to 83°", "strike_type": "between", "floor_strike": 82, "cap_strike": 83},
    {"title": "High temperature in NYC on Jun 3?", "yes_sub_title": "79° or below", "strike_type": "less", "cap_strike": 80}]}
g = kalshi_group(temps)
test_eq(g["statements"][0], "High temperature in NYC on Jun 3: 80° to 81°")
test_eq(sorted(g["pairs"]), [(0, 1, "rules out"), (0, 2, "rules out"), (1, 2, "rules out")])
pm = {"event_ticker": "KXPM", "mutually_exclusive": True, "markets": [{"title": f"Will {n} be the next Prime Minister?", "strike_type": "custom"} for n in ("Ana", "Ion", "Dan")]}
test_eq(kalshi_group(pm)["kind"], "exclusive")

In [ ]:
#| export
_UP = re.compile(r"\b(above|over|more than|greater than|higher than|at least|reach|reaches|hit|hits|exceed|exceeds|surpass|top|or more|or higher)\b", re.I)
_DOWN = re.compile(r"\b(below|under|less than|fewer than|lower than|at most|dip to|dips to|fall to|falls to|drop to|drops to|or less|or lower)\b", re.I)
_MONTHS = {m: i for i, m in enumerate("january february march april may june july august september october november december".split(), 1)}

def _number(tok:str) -> float|None:
    m = re.fullmatch(r"[$€£]?(-?[\d,]*\.?\d+)([kKmMbB]?)%?[?.,]?", tok)
    if not m: return None
    v = float(m[1].replace(",", "")); return v * {"k": 1e3, "m": 1e6, "b": 1e9}.get(m[2].lower(), 1)

def ladder_pairs(questions:list) -> list|None:
    "The implications among questions that differ in one number (read with the template's direction: above, below) or one date (with 'by' or 'before'); None when they are no ladder"
    toks = [q.split() for q in questions]
    if len({len(t) for t in toks}) != 1 or len(questions) < 2: return None
    diff = [i for i in range(len(toks[0])) if len({t[i] for t in toks}) > 1]
    vals = [_number(t[diff[0]]) for t in toks] if len(diff) == 1 else [None]
    if None not in vals:                                   # one number differs
        if len(set(vals)) < len(vals): return None
        tpl = " ".join(w for k, w in enumerate(toks[0]) if k != diff[0])
        up, down = bool(_UP.search(tpl)), bool(_DOWN.search(tpl))
        if up == down: return None
        return [(a, b, "implies") for a in range(len(vals)) for b in range(len(vals)) if a != b and (vals[a] > vals[b] if up else vals[a] < vals[b])]
    if len(diff) in (1, 2) and re.search(r"\b(by|before)\b", " ".join(toks[0]), re.I):     # a date: a month, and perhaps its day
        def when(t):
            words = [t[i].strip(",.?").lower() for i in diff]
            month = next((_MONTHS[w] for w in words if w in _MONTHS), None)
            day = next((int(w) for w in words if w.isdigit()), 31 if len(diff) == 1 else None)
            return None if month is None or day is None else (month, day)
        ds = [when(t) for t in toks]
        if None in ds or len(set(ds)) < len(ds): return None
        return [(a, b, "implies") for a in range(len(ds)) for b in range(len(ds)) if a != b and ds[a] < ds[b]]
    return None

def polymarket_group(e:dict, most:int=8, rng=None, source:str="polymarket") -> dict|None:
    "A group from a Polymarket event (with its markets): an exclusive group when it is `negRisk`, else a ladder when its questions differ in one number or date; None otherwise"
    qs = _unique((m.get("question") or "").strip() for m in e.get("markets") or [] if (m.get("question") or "").strip())
    qs = _sample(qs, most, rng)
    if len(qs) < 2: return None
    if e.get("negRisk"): pairs, kind = [(a, b, "rules out") for a in range(len(qs)) for b in range(a + 1, len(qs))], "exclusive"
    elif (pairs := ladder_pairs(qs)) is not None: kind = "ladder"
    else: return None
    return group(qs, pairs, {str(e.get("id", ""))}, source, f"{source}/{e.get('id', '')}", kind=kind, event=e.get("id"), category=e.get("category"))

def manifold_group(m:dict, most:int=8, rng=None, source:str="manifold") -> dict|None:
    "A group from a Manifold multiple-choice market whose answers sum to one: each answer, written after its question, rules out the others; None otherwise"
    if m.get("outcomeType") != "MULTIPLE_CHOICE" or not m.get("shouldAnswersSumToOne"): return None
    q = (m.get("question") or "").strip()
    ans = _sample(_unique((a.get("text") or "").strip() for a in m.get("answers") or [] if (a.get("text") or "").strip()), most, rng)
    if not q or len(ans) < 2: return None
    st = [f"{q} {a}" for a in ans]
    return group(st, [(a, b, "rules out") for a in range(len(st)) for b in range(a + 1, len(st))], {m.get("id", "")}, source, f"{source}/{m.get('id', '')}",
                 kind="exclusive", event=m.get("id"))

In [ ]:
test_eq(ladder_pairs(["Will Bitcoin reach $100,000 in May?", "Will Bitcoin reach $120,000 in May?"]), [(1, 0, "implies")])
test_eq(ladder_pairs(["Will Bitcoin dip to $80k in May?", "Will Bitcoin dip to $70k in May?"]), [(1, 0, "implies")])
test_eq(ladder_pairs(["Ceasefire by March 31?", "Ceasefire by June 30?"]), [(0, 1, "implies")])
test_eq(ladder_pairs(["Ceasefire by June 2025?", "Ceasefire by March 2025?"]), [(1, 0, "implies")])      # only the month differs
test_eq(ladder_pairs(["Will Spain win?", "Will Brazil win?"]), None)
g = polymarket_group({"id": 7, "negRisk": True, "markets": [{"question": "Will Spain win the World Cup?"}, {"question": "Will Brazil win the World Cup?"}]})
test_eq((g["kind"], g["pairs"]), ("exclusive", [(0, 1, "rules out")]))
g = manifold_group({"id": "m1", "outcomeType": "MULTIPLE_CHOICE", "shouldAnswersSumToOne": True, "question": "Who will win the 2024 US election?",
                    "answers": [{"text": "Kamala Harris"}, {"text": "Donald Trump"}]})
test_eq(g["statements"], ["Who will win the 2024 US election? Kamala Harris", "Who will win the 2024 US election? Donald Trump"])

The exchanges' public APIs give events with their markets, no account needed. `fetch_json` retries and paces the calls; the three fetchers page through events (Kalshi, Polymarket) or multiple-choice markets (Manifold, whose answers need one call per market). Paging by offset stops somewhere: Polymarket refuses an offset past about 3,000 (deeper needs its keyset endpoint) and Manifold past 1,000, so a fetcher keeps what it has when the API refuses the next page.

In [ ]:
#| export
def fetch_json(url:str, params:dict|None=None, tries:int=4, pause:float=0.2):
    "GET a JSON API, with a few retries (a request the API refuses, 4xx, is not retried)"
    full = url + ("?" + urllib.parse.urlencode(params) if params else "")
    for k in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(full, headers={"User-Agent": "sysone"}), timeout=60) as r:
                time.sleep(pause); return json.loads(r.read())
        except urllib.error.HTTPError as e:
            if 400 <= e.code < 500 or k == tries - 1: raise
            time.sleep(2 ** k)
        except Exception:
            if k == tries - 1: raise
            time.sleep(2 ** k)

def _pages(get, n:int) -> list:
    "Items from `get(offset)` until `n` are found, a page comes back empty, or the API refuses the next page (keeping what came before)"
    out = []
    while len(out) < n:
        try: page = get(len(out))
        except urllib.error.HTTPError:
            if out: break
            raise
        if not page: break
        out += page
    return out[:n]

KALSHI, POLYMARKET, MANIFOLD = "https://api.elections.kalshi.com/trade-api/v2", "https://gamma-api.polymarket.com", "https://api.manifold.markets/v0"

def kalshi_events(n:int, status:str="settled") -> list:
    "Up to `n` Kalshi events of `status` with their markets"
    out, cursor = [], None
    while len(out) < n:
        r = fetch_json(f"{KALSHI}/events", {"limit": 200, "status": status, "with_nested_markets": "true"} | ({"cursor": cursor} if cursor else {}))
        out += r.get("events") or []; cursor = r.get("cursor")
        if not cursor or not r.get("events"): break
    return out[:n]

def polymarket_events(n:int, closed:bool=True) -> list:
    "Up to `n` Polymarket events with their markets, the most traded first"
    return _pages(lambda off: fetch_json(f"{POLYMARKET}/events", {"closed": str(closed).lower(), "limit": 100, "offset": off, "order": "volume",
                                                                  "ascending": "false"}), n)

def manifold_markets(n:int) -> list:
    "Up to `n` Manifold multiple-choice markets, each with its answers"
    found = _pages(lambda off: fetch_json(f"{MANIFOLD}/search-markets", {"term": "", "contractType": "MULTIPLE_CHOICE", "limit": 100, "offset": off,
                                                                          "sort": "most-popular"}), n)
    return [fetch_json(f"{MANIFOLD}/market/{m['id']}", pause=0.15) for m in found]

In [ ]:
def refuses(off):
    if off >= 200: raise urllib.error.HTTPError("u", 422, "offset too large", None, None)
    return [off + k for k in range(100)]
test_eq(len(_pages(refuses, 1000)), 200)                           # the API stops at 200: what came before is kept

## Lists from groups

`group_lists` joins a few groups into a list, shuffled together, when they share no key and no name: a Kalshi event about New York's temperature and a Polymarket event about the World Cup share nothing that could relate them, while two events about Bitcoin might. Every relation in the list is then one a group fixed, and every other pair is unrelated. `split_groups` sets groups aside for testing by the hash of their keys, so a base question and everything built from it fall on one side.

In [ ]:
#| export
def split_groups(groups:list, test:float=0.1, seed:int=0) -> tuple:
    "Training and test groups: a group is a test group when any of its keys hashes into the test share"
    hit = lambda k: int(hashlib.sha1(f"{seed}:{k}".encode()).hexdigest()[:8], 16) / 16**8 < test
    tr, te = [], []
    for g in groups: (te if any(hit(k) for k in g["keys"]) else tr).append(g)
    return tr, te

def group_lists(groups:list, rng:random.Random, size:tuple=(1, 3), words:int=220, asked=("implies", "rules out"), source:str|None=None,
                suite:str|None=None, tries:int=30) -> list:
    "Lists of statements, each a few groups that share no key and no name, so that every pair is related as a group says or unrelated"
    pool, out = list(groups), []
    rng.shuffle(pool)
    while pool:
        cur = [pool.pop()]; keys, nm = set(cur[0]["keys"]), set(cur[0]["names"]); n = sum(len(s.split()) for s in cur[0]["statements"])
        want = rng.randint(*size)
        for _ in range(tries):
            if len(cur) >= want or not pool: break
            k = rng.randrange(len(pool)); g = pool[k]; m = sum(len(s.split()) for s in g["statements"])
            if keys & g["keys"] or nm & g["names"] or n + m > words: continue
            cur.append(pool.pop(k)); keys |= g["keys"]; nm |= g["names"]; n += m
        st, pairs = [], []
        for g in cur:
            b = len(st); st += g["statements"]; pairs += [(i + b, j + b, r) for i, j, r in g["pairs"]]
        src = source or "+".join(_unique(g["source"] for g in cur))
        out.append(statement_list(st, pairs, rng, list(asked), src, f"{src}/{len(out)}:{cur[0]['id']}", suite) | {"groups": [g["id"] for g in cur]})
    return out

In [ ]:
gs = paleka_groups(cons, "Consequence") + [kalshi_group(temps), kalshi_group(pm)] + paleka_groups(andt, "And")
ls = group_lists(gs, random.Random(0), size=(3, 3))
assert all(spans_ok(gold_example(r)) for r in ls)
qs_of = lambda r: {re.sub(r"^(\d+\. |- )", "", q) for q in r["text"].splitlines()}
P, Q = "Will Haley be the Republican nominee?", "Will DeSantis be the Republican nominee?"
test_eq([r for r in ls if {P, Q} <= qs_of(r)], [])                 # P and Q of one tuple never meet in a list
test_eq(sum(len(r["groups"]) for r in ls), len(gs))
print(ls[0]["text"])

1. High temperature in NYC on Jun 3: 79° or below
2. Will both Haley and DeSantis be the Republican nominees?
3. Will the 100m winning time break 9.70s?
4. High temperature in NYC on Jun 3: 82° to 83°
5. Will Haley be the Republican nominee?
6. High temperature in NYC on Jun 3: 80° to 81°
7. Will the 100m winning time break 9.63s?


## Generated markets, from their outcomes

The generator in [GLiNER multitask](33_gliner_multitask.ipynb#statements-and-how-they-relate) wrote each relation of a market group by hand, and its lists held at most one group of each kind. Over a pool of real-looking markets the trained model then related questions of the same form about different things, as if "the S&P 500 below 5,400" implied "Bitcoin below $70,000": nothing it trained on had asked it to check what a question is about.

This generator states what can happen instead. A market group has *worlds*, the outcomes it can resolve to (which bucket a price ends in, who wins an election, how far a team goes), and each question is true in some of them. Then one question *implies* another when every world where the first is true makes the second true, and *rules out* it when no world makes both true; questions on different groups share no world and stay unrelated. That stays exact for everything the lists now hold:

- **several groups of one kind** (prices of three assets, or of one asset on two dates), the hard negatives the model lacked;
- **negations** ("Will Arsenal not beat Chelsea?"), **or** and **and** between two questions;
- **more ways to ask** each kind (close, finish, settle above; win, be elected in).

In [ ]:
#| export
@dataclass
class Fact:
    "A market question: a clause ('Bitcoin close above $90,000 on June 30, 2025'), the groups it reads, and when it is true (`holds`, given a world for each)"
    clause:str
    groups:tuple
    holds:object
    stage:tuple|None = None                 # (group, team, stage) for tournament rounds: what 'is a precondition of' reads
    negate:object = None                    # the clause of its negation, when it has one
    @property
    def text(self) -> str: return f"Will {self.clause}?"

def _neg(f:Fact) -> Fact:
    return Fact(f.negate, f.groups, lambda w, h=f.holds: not h(w), None, None)

def _both(a:Fact, b:Fact) -> Fact: return Fact(f"{a.clause} and {b.clause}", tuple(_unique(a.groups + b.groups)), lambda w: a.holds(w) and b.holds(w))
def _either(a:Fact, b:Fact) -> Fact: return Fact(f"{a.clause} or {b.clause}", tuple(_unique(a.groups + b.groups)), lambda w: a.holds(w) or b.holds(w))

def _world_group(kind:str, rng, w:dict, part:str, used:set):
    "One market group of `kind`: ({group: its worlds}, its questions); None when what it is about is taken"
    if kind == "thresholds":
        a, unit, levels = rng.choice(_part(w["assets"], part)); d = rng.choice(w["dates"])
        if (a, d) in used or ("reach", a) in used: return None
        used |= {(a, d), ("asset", a)}; g = f"price:{a}:{d}"
        xs = sorted(rng.sample(levels, rng.randint(2, min(4, len(levels))))); verb = rng.choice(["close", "be", "finish", "settle"])
        facts = []
        for i, x in enumerate(xs):
            for side in ("above", "below"):
                if rng.random() < 0.55:
                    hold = (lambda wd, i=i: wd[g] > i) if side == "above" else (lambda wd, i=i: wd[g] <= i)
                    facts.append(Fact(f"{a} {verb} {side} {_money(x, unit)} on {d}", (g,), hold, negate=f"{a} not {verb} {side} {_money(x, unit)} on {d}"))
        return {g: list(range(len(xs) + 1))}, facts or None
    if kind == "deadlines":
        h = rng.choice(_part(w["happenings"], part))
        if h in used: return None
        used.add(h); g = f"when:{h}"; ds = sorted(rng.sample(range(len(w["dates"])), rng.randint(2, 3)))
        word = rng.choice(["by", "no later than"])
        return {g: list(range(len(ds) + 1))}, [Fact(f"{h} {word} {w['dates'][d]}", (g,), lambda wd, j=j: wd[g] <= j) for j, d in enumerate(ds)]
    if kind == "matches":
        name, unit, margins, teams = rng.choice(w["leagues"]); t1, t2 = rng.sample(_part(teams, part), 2)
        if frozenset((t1, t2)) in used: return None
        used.add(frozenset((t1, t2))); g = f"match:{t1}:{t2}"
        ks = sorted(rng.sample(margins, rng.randint(0, min(2, len(margins))))); cuts = [1] + ks
        worlds = ["t2"] + (["draw"] if unit == "goals" else []) + list(range(len(cuts)))      # t1's margin: at least cuts[k], below cuts[k + 1]
        verb = rng.choice(["beat", "defeat"])
        t1wins = lambda wd: isinstance(wd[g], int)
        facts = [Fact(f"{t1} {verb} {t2} in {name}", (g,), t1wins, negate=f"{t1} not {verb} {t2} in {name}")]
        facts += [Fact(f"{t1} {verb} {t2} by {k} or more {unit} in {name}", (g,), lambda wd, m=m: isinstance(wd[g], int) and wd[g] >= m,
                       negate=f"{t1} not {verb} {t2} by {k} or more {unit} in {name}") for m, k in enumerate(cuts) if m > 0]
        if rng.random() < 0.6: facts.append(Fact(f"{t2} {verb} {t1} in {name}", (g,), lambda wd: wd[g] == "t2", negate=f"{t2} not {verb} {t1} in {name}"))
        return {g: worlds}, facts
    if kind == "elections":
        e, people = rng.choice(w["elections"])
        if e in used: return None
        pool = _part(people, part); used.add(e); g = f"election:{e}"
        cs = rng.sample(pool, rng.randint(2, min(3, len(pool)))); party = dict(pool)
        verb = rng.choice(["win", "be elected in"])
        facts = [Fact(f"{c} {verb} {e}", (g,), lambda wd, c=c: wd[g] == c, negate=f"{c} not {verb} {e}") for c, _ in cs]
        facts += [Fact(f"the {p} candidate win {e}", (g,), lambda wd, p=p: party.get(wd[g]) == p, negate=f"the {p} candidate not win {e}")
                  for p in _unique(p for _, p in cs) if rng.random() < 0.6]
        return {g: [c for c, _ in pool] + ["someone else"]}, facts
    if kind == "stages":
        name, teams, stages = rng.choice(w["tournaments"])
        if name in used: return None
        used.add(name); g = f"cup:{name}"; pool = _part(teams, part); t = rng.choice(pool); other = rng.choice([x for x in pool if x != t])
        last = len(stages) - 1
        worlds = [(r, c) for r in range(-1, last + 1) for c in (t, other, "someone else") if (r == last) == (c == t)]
        ss = sorted(rng.sample(range(len(stages)), rng.randint(2, min(3, len(stages)))))
        facts = [Fact(f"{t} {stages[s]} {name}", (g,), lambda wd, s=s: wd[g][0] >= s, stage=(g, t, s), negate=f"{t} not {stages[s]} {name}") for s in ss]
        if ss[-1] == last and rng.random() < 0.5:
            facts.append(Fact(f"{other} {stages[last]} {name}", (g,), lambda wd: wd[g][1] == other, negate=f"{other} not {stages[last]} {name}"))
        return {g: worlds}, facts
    if kind == "reach":
        a, unit, levels = rng.choice(_part(w["assets"], part))
        if ("asset", a) in used or ("reach", a) in used: return None
        used |= {("asset", a), ("reach", a)}; g = f"reach:{a}"
        xs = sorted(rng.sample(levels, rng.randint(1, min(3, len(levels))))); ds = sorted(rng.sample(range(len(w["dates"])), rng.randint(1, 3)))
        worlds = [p for p in product(range(len(xs) + 1), repeat=len(ds)) if all(p[k] <= p[k + 1] for k in range(len(p) - 1))]   # highest level passed by each date
        combos = rng.sample([(i, j) for i in range(len(xs)) for j in range(len(ds))], min(len(xs) * len(ds), rng.randint(2, 4)))
        if len(combos) < 2: return None
        verb = rng.choice(["reach", "hit"])
        return {g: worlds}, [Fact(f"{a} {verb} {_money(xs[i], unit)} by {w['dates'][ds[j]]}", (g,), lambda wd, i=i, j=j: wd[g][j] > i,
                                  negate=f"{a} not {verb} {_money(xs[i], unit)} by {w['dates'][ds[j]]}") for i, j in combos]
    if kind == "counts":
        subj, what, levels = rng.choice(_part(w["counts"], part))
        if subj in used: return None
        used.add(subj); g = f"count:{subj}"; ns = sorted(rng.sample(levels, rng.randint(2, min(3, len(levels)))))
        facts = [Fact(f"{subj} {side} {x:,} {what}", (g,), (lambda wd, i=i: wd[g] > i) if side == "at least" else (lambda wd, i=i: wd[g] <= i))
                 for i, x in enumerate(ns) for side in ("at least", "fewer than") if rng.random() < 0.6]
        return {g: list(range(len(ns) + 1))}, facts or None
    if kind == "ranges":
        q, unit, lo, step, k, res = rng.choice(_part(w["ranges"], part))
        if q in used: return None
        used.add(q); g = f"range:{q}"
        fmt = lambda x: f"{x:.1f}%" if unit == "%" else f"{round(x):,}"
        bounds = [round(lo + i * step, 6) for i in range(k)]                  # cells: below bounds[0], then [bounds[i], bounds[i + 1]), then above the last
        cells = sorted(rng.sample(range(k - 1), rng.randint(2, min(3, k - 1)))); ats = sorted(rng.sample(range(1, k), rng.randint(1, 2)))
        facts = [Fact(f"{q} be between {fmt(bounds[c])} and {fmt(bounds[c + 1] - res)}", (g,), lambda wd, c=c: wd[g] == c + 1) for c in cells]
        facts += [Fact(f"{q} be at least {fmt(bounds[a])}", (g,), lambda wd, a=a: wd[g] >= a + 1) for a in ats]
        return {g: list(range(k + 1))}, facts
    raise ValueError(f"no market kind {kind!r}")

def fact_pairs(facts:list, worlds:dict) -> list:
    "Every relation among `facts`, from their worlds: implies (true wherever the first is), rules out (never true together), is a precondition of (an earlier round of the same team)"
    out = []
    for i, a in enumerate(facts):
        for j, b in enumerate(facts):
            if i == j or not set(a.groups) & set(b.groups): continue
            gs = _unique(tuple(a.groups) + tuple(b.groups)); seen_a = seen_b = both = a_only = False
            for combo in product(*[worlds[x] for x in gs]):
                wd = dict(zip(gs, combo)); x, y = a.holds(wd), b.holds(wd)
                seen_a |= x; seen_b |= y; both |= x and y; a_only |= x and not y
            if seen_a and not a_only: out.append((i, j, "implies"))
            if seen_a and seen_b and not both and i < j: out.append((i, j, "rules out"))
            if a.stage and b.stage and a.stage[:2] == b.stage[:2] and a.stage[2] < b.stage[2]: out.append((i, j, "is a precondition of"))
    return out

def _satisfiable(f:Fact, worlds:dict) -> bool:
    "Whether a question can resolve either way"
    vals = {f.holds(dict(zip(f.groups, c))) for c in product(*[worlds[g] for g in f.groups])}
    return vals == {True, False}

In [ ]:
#| export
def market_lists(n:int, seed:int=0, part:str="train", kinds:tuple=MARKET_KINDS[:-1], world:dict=MARKET_WORLD, same_kind:float=0.7,
                 negate:float=0.2, combine:float=0.3, most:int=10, source:str="markets-v2", suite:str|None=None) -> list:
    "`n` lists of market questions: one to three groups (each of the first one's kind with chance `same_kind`), up to two questions about other things, some negated, sometimes two joined by 'and' or 'or'; every relation that holds marked, from the groups' worlds"
    rng, out = random.Random(f"v2:{seed}:{part}"), []
    asked = ["implies", "rules out", "is a precondition of"]
    while len(out) < n:
        used, worlds, facts, ks = set(), {}, [], []
        first = rng.choice(list(kinds))
        for _ in range(rng.randint(1, 3)):
            kind = first if rng.random() < same_kind else rng.choice(list(kinds))
            got = _world_group(kind, rng, world, part, used)
            if got and got[1]: worlds |= got[0]; facts += got[1]; ks.append(kind)
        for _ in range(rng.randint(0, 2)):
            got = _world_group(rng.choice(list(kinds)), rng, world, part, used)
            if got and got[1]: worlds |= got[0]; facts.append(rng.choice(got[1]))
        if not facts: continue
        for f in [f for f in facts if f.negate and rng.random() < negate]:
            if rng.random() < 0.5: facts.append(_neg(f))
            else: facts[facts.index(f)] = _neg(f)
        if len(facts) >= 2 and rng.random() < combine:
            a, b = rng.sample(facts, 2); facts.append((_both if rng.random() < 0.5 else _either)(a, b))
        seen, keep = set(), []
        for f in facts:
            if f.text not in seen and _satisfiable(f, worlds): seen.add(f.text); keep.append(f)
        if len(keep) < 2 or len(keep) > most: continue
        out.append(statement_list([f.text for f in keep], fact_pairs(keep, worlds), rng, asked, source, f"{source}-{part}/{len(out)}", suite)
                   | {"market_kinds": ks})
    return out

def market_world_pool(groups:int, seed:int=0, part:str="test", kinds:tuple=MARKET_KINDS, world:dict=MARKET_WORLD, negate:float=0.15) -> tuple:
    "Questions from up to `groups` market groups (kinds drawn with repeats, so several groups of one kind), some negated, shuffled; and every relation among them, a symmetric one both ways"
    rng, used, worlds, facts, made = random.Random(f"v2pool:{seed}:{part}"), set(), {}, [], 0
    for _ in range(groups * 6):
        got = _world_group(rng.choice(list(kinds)), rng, world, part, used)
        if not got or not got[1]: continue
        worlds |= got[0]; facts += got[1]; made += 1
        if made == groups: break
    facts = [(_neg(f) if f.negate and rng.random() < negate else f) for f in facts]
    facts = [f for f in facts if _satisfiable(f, worlds)]
    rng.shuffle(facts)
    pairs = fact_pairs(facts, worlds)
    gold = {(i, j, r) for i, j, r in pairs} | {(j, i, r) for i, j, r in pairs if r == "rules out"}
    return [f.text for f in facts], sorted(gold)

The relations come out of the worlds, so the hand-written logic of the first generator is a check on them: above a higher price implies above a lower one, a candidate implies their party and rules out the others, winning a cup implies reaching its final, which is its precondition. A negation and its question rule each other out; "A and B" implies A; A implies "A or B"; questions about different things are unrelated.

In [ ]:
def facts_of(kind, seed=0, part="train"):
    for s in range(seed, seed + 200):
        got = _world_group(kind, random.Random(s), MARKET_WORLD, part, set())
        if got and got[1] and len(got[1]) >= 2: return got
el_w, el = facts_of("elections")
rel = lambda fs, w: {(fs[i].clause, fs[j].clause, r) for i, j, r in fact_pairs(fs, w)}
cands = [f for f in el if "candidate" not in f.clause]
assert all((a.clause, b.clause, "rules out") in rel(el, el_w) for a in cands for b in cands if el.index(a) < el.index(b))
cup_w, cup = facts_of("stages", 3)
r = rel(cup, cup_w)
assert any(rr == "is a precondition of" for *_, rr in r) and all(((b, a, "implies") in r) for a, b, rr in r if rr == "is a precondition of")
pr_w, pr = facts_of("thresholds", 5)
f, nf = pr[0], _neg(pr[0])
test_eq(fact_pairs([f, nf], pr_w), [(0, 1, "rules out")])
x_w, x = facts_of("counts", 2); y_w, y = facts_of("deadlines", 4)
ws = x_w | y_w
test_eq(fact_pairs([x[0], y[0]], ws), [])                                                  # different things: unrelated
test_eq(sorted(fact_pairs([x[0], y[0], _both(x[0], y[0]), _either(x[0], y[0])], ws)),
        [(0, 3, "implies"), (1, 3, "implies"), (2, 0, "implies"), (2, 1, "implies"), (2, 3, "implies")])

In [ ]:
lists = market_lists(300, seed=0)
assert all(spans_ok(gold_example(r)) for r in lists)
repeat = [r for r in lists if len(r["market_kinds"]) > len(set(r["market_kinds"]))]
print(f"{len(repeat)} of 300 lists hold two or more groups of one kind; one:")
print(next(r for r in repeat if r["market_kinds"].count("thresholds") >= 2)["text"])
assert len(repeat) > 100 and any(" not " in r["text"] for r in lists) and any(" or " in r["text"] for r in lists)

109 of 300 lists hold two or more groups of one kind; one:
1. Will gold finish above $2,600 on March 31, 2026?
2. Will XRP be above $1 on December 31, 2025?
3. Will gold not finish below $2,600 on March 31, 2026?
4. Will XRP be above $4 on December 31, 2025?
5. Will gold not finish above $2,800 on March 31, 2026?
6. Will XRP be above $3 on December 31, 2025?
7. Will XRP be above $1 on December 31, 2025 and XRP be above $3 on December 31, 2025?
8. Will XRP be above $5 on December 31, 2025?
9. Will gold finish below $2,200 on March 31, 2026?
10. Will the European Commission fine Google no later than June 30, 2026?


In [ ]:
qs, gold = market_world_pool(15, seed=0)
print(len(qs), "questions,", len(gold), "relations")
assert len(qs) > 25 and all(0 <= i < len(qs) and 0 <= j < len(qs) for i, j, _ in gold)
pairs_by = Counter(r for *_, r in gold); assert pairs_by["implies"] and pairs_by["rules out"]

50 questions, 85 relations


The test half of the names (assets, teams, candidates, events) stays out of training lists, as in the first generator, and so does a kind of market, ranges, unless asked for.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()